# Machine Condition Classification

Kaggle competition: industrial-machine-condition-classification-mtech

Predict the condition of a machine from its sensor readings.
- 0 = Normal
- 1 = Thermal Failure
- 2 = Mechanical Failure

Metric is accuracy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from sklearn.model_selection import KFold, cross_validate, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.base import clone
from xgboost import XGBClassifier

SEED = 42

In [ ]:
path = "/kaggle/input/competitions/industrial-machine-condition-classification-mtech/"
train = pd.read_csv(path + "student_dataset.csv")
test = pd.read_csv(path + "test.csv")
sample_sub = pd.read_csv(path + "sample_submission.csv")

print(train.shape, test.shape, sample_sub.shape)
train.head()

In [ ]:
train.info()

In [ ]:
# missing values in train and test
missing = pd.DataFrame({
    "train": train.isna().sum(),
    "train_%": (train.isna().mean() * 100).round(2),
    "test": test.isna().sum(),
    "test_%": (test.isna().mean() * 100).round(2),
})
print(missing)

In [ ]:
print(train["condition"].value_counts())

## Preprocessing
unit_type is text so map it to numbers. Then fill the missing values with the mean (very few are missing, less than 0.5% per column).

In [ ]:
type_map = {"S": 0, "M": 1, "H": 2}
train["unit_type"] = train["unit_type"].map(type_map)
test["unit_type"] = test["unit_type"].map(type_map)

feature_cols = [c for c in train.columns if c not in ["unit_id", "condition"]]

imp = SimpleImputer(strategy="mean")
train[feature_cols] = imp.fit_transform(train[feature_cols])
test[feature_cols] = imp.transform(test[feature_cols])

# unit_type should stay 0/1/2
train["unit_type"] = train["unit_type"].round().astype(int)
test["unit_type"] = test["unit_type"].round().astype(int)

print(train[feature_cols].isna().sum().sum(), test[feature_cols].isna().sum().sum())

## EDA

In [ ]:
class_names = {0: "Normal", 1: "Thermal Failure", 2: "Mechanical Failure"}
counts = train["condition"].value_counts()

plt.figure(figsize=(7, 4))
sns.barplot(x=[class_names[i] for i in counts.index], y=counts.values)
plt.title("Distribution of Machine Condition")
plt.ylabel("Number of machines")
plt.show()

In [ ]:
num_cols = [c for c in feature_cols if c != "unit_type"]
print(train[num_cols].describe().T.round(2))

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(train[col], bins=40, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel("")
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# outliers with IQR rule
# also check how many of the outliers are failures
rows = []
for col in num_cols:
    q1, q3 = train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (train[col] < q1 - 1.5 * iqr) | (train[col] > q3 + 1.5 * iqr)
    fail = (train.loc[mask, "condition"] != 0).mean() * 100 if mask.sum() > 0 else np.nan
    rows.append([col, mask.sum(), round(mask.mean() * 100, 2), round(fail, 1)])

outliers = pd.DataFrame(rows, columns=["feature", "n_outliers", "outlier_%", "failure_%"])
print(outliers.sort_values("n_outliers", ascending=False))

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(x=train[col], ax=axes[i], color="lightcoral")
    axes[i].set_title(col)
    axes[i].set_xlabel("")
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

# most outliers are failures so we keep them

In [ ]:
# features vs condition
plot_df = train.copy()
plot_df["cond"] = plot_df["condition"].map(class_names)
order = ["Normal", "Thermal Failure", "Mechanical Failure"]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(data=plot_df, x="cond", y=col, order=order, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", labelsize=8)
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

print(plot_df.groupby("cond")[num_cols].median().T[order].round(2))

# unit_type vs condition (% per type)
print(pd.crosstab(plot_df["unit_type"].map({0: "S", 1: "M", 2: "H"}), plot_df["cond"], normalize="index")[order].round(3) * 100)

In [ ]:
corr = train[num_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation")
plt.show()

# pairs with |r| > 0.8
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().reset_index()
pairs.columns = ["f1", "f2", "r"]
print(pairs[pairs["r"].abs() > 0.8].sort_values("r", ascending=False))

In [ ]:
# anova f score - which features separate the 3 classes the most
f, p = f_classif(train[num_cols], train["condition"])
anova = pd.DataFrame({"feature": num_cols, "F": f.round(1)}).sort_values("F", ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=anova, y="feature", x="F", color="steelblue")
plt.title("ANOVA F-score")
plt.show()
print(anova)

EDA notes:
- 50% normal, 25% thermal, 25% mechanical. so baseline accuracy is 50%
- outliers are mostly failures, kept them
- thermal failures -> high process_temp, thermal_stress
- mechanical failures -> high component_wear, vibration, mechanical_load
- speed/torque/power/temp features are highly correlated

## Models

In [ ]:
FEATURES = ["unit_type", "ambient_temp", "process_temp", "coolant_flow",
            "shaft_speed", "drive_torque", "power_load", "component_wear",
            "thermal_stress", "mechanical_load", "vibration_index"]

X = train[FEATURES]
y = train["condition"]

# same 5 folds for every model
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(random_state=SEED),
    "XGBoost": XGBClassifier(random_state=SEED),
}

results = []
for name, model in models.items():
    cv = cross_validate(model, X, y, cv=kf, scoring="accuracy", return_train_score=True)
    results.append([name, cv["train_score"].mean(), cv["test_score"].mean(), cv["test_score"].std()])
    print(name, round(cv["test_score"].mean(), 4))

results = pd.DataFrame(results, columns=["model", "train_acc", "val_acc", "val_std"])
results["gap"] = results["train_acc"] - results["val_acc"]
results = results.sort_values("val_acc", ascending=False)
print(results.round(4))

plt.figure(figsize=(9, 4.5))
sns.barplot(data=results, x="val_acc", y="model", color="steelblue")
plt.axvline(0.5, color="red", linestyle="--", label="baseline 50%")
plt.legend()
plt.title("5-fold CV accuracy")
plt.show()

In [ ]:
# where does xgboost make mistakes
pred = cross_val_predict(XGBClassifier(random_state=SEED), X, y, cv=kf)

labels = ["Normal", "Thermal", "Mechanical"]
ConfusionMatrixDisplay.from_predictions(y, pred, display_labels=labels, cmap="Blues")
plt.show()
print(classification_report(y, pred, target_names=labels, digits=4))

## Tuning XGBoost

In [ ]:
param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
}

grid = GridSearchCV(XGBClassifier(random_state=SEED), param_grid, cv=kf, scoring="accuracy")
grid.fit(X, y)
print(grid.best_params_, round(grid.best_score_, 4))

## Feature engineering
wear_torque -> worn parts + high torque (mechanical)

stress_per_flow -> high stress with low cooling (thermal)

In [ ]:
for df in [train, test]:
    df["wear_torque"] = df["component_wear"] * df["drive_torque"]
    df["stress_per_flow"] = df["thermal_stress"] / df["coolant_flow"]

xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.1, random_state=SEED)

sets = {
    "baseline": FEATURES,
    "+ wear_torque": FEATURES + ["wear_torque"],
    "+ stress_per_flow": FEATURES + ["stress_per_flow"],
    "+ both": FEATURES + ["wear_torque", "stress_per_flow"],
}
for name, feats in sets.items():
    s = cross_val_score(xgb, train[feats], y, cv=kf, scoring="accuracy")
    print(name, round(s.mean(), 4), round(s.std(), 4))

In [ ]:
# gain is small, so check it again with 5 different seeds (25 folds)
both = FEATURES + ["wear_torque", "stress_per_flow"]
diffs = []
for seed in range(5):
    for tr, va in KFold(5, shuffle=True, random_state=seed).split(train):
        a = clone(xgb).fit(train[FEATURES].iloc[tr], y.iloc[tr]).score(train[FEATURES].iloc[va], y.iloc[va])
        b = clone(xgb).fit(train[both].iloc[tr], y.iloc[tr]).score(train[both].iloc[va], y.iloc[va])
        diffs.append(b - a)

diffs = np.array(diffs)
print("mean gain:", round(diffs.mean(), 4))
print("both wins:", (diffs > 0).sum(), "| baseline wins:", (diffs < 0).sum(), "| ties:", (diffs == 0).sum())

In [ ]:
# keep both new features and tune again
FEATURES = both
X = train[FEATURES]

grid = GridSearchCV(XGBClassifier(random_state=SEED), param_grid, cv=kf, scoring="accuracy")
grid.fit(X, y)
print(grid.best_params_, round(grid.best_score_, 4))

res = pd.DataFrame(grid.cv_results_)[["params", "mean_test_score", "std_test_score"]]
print(res.sort_values("mean_test_score", ascending=False).round(4))

## Final model and submission

In [ ]:
final_model = grid.best_estimator_   # already refit on all data

imp_ = pd.Series(final_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plt.figure(figsize=(9, 5.5))
sns.barplot(x=imp_.values, y=imp_.index, color="steelblue")
plt.title("XGBoost feature importance")
plt.show()
print(imp_.round(4))

In [ ]:
test_pred = final_model.predict(test[FEATURES])

sub = pd.DataFrame({"unit_id": test["unit_id"], "condition": test_pred})
sub = sample_sub[["unit_id"]].merge(sub, on="unit_id", how="left")

print(sub.shape)
print(sub["condition"].value_counts(normalize=True).round(3))
sub.to_csv("submission.csv", index=False)
sub.head()